This note is a cleaned-up piece of my ongoing neural-network exercises. Before training a network, I wanted to understand what PyTorch differentiates when I call `backward()` and check the result against the chain rule by hand.

## A small graph

Start with the scalar result

$$y = \frac{1}{n}\sum_{i=1}^{n}\left[(x_i+2)^2+3\right].$$

The shift, square, addition, and mean are separate steps. For $x_i$, the derivative is $2(x_i+2)/n$: the constant 3 disappears, and the mean contributes the factor $1/n$.

In [1]:
import torch

x = torch.arange(3, dtype=torch.float64, requires_grad=True)
y = ((x + 2).square() + 3).mean()
print(f"x = {x.detach().tolist()}")
print(f"y = {y.item():.4f}")

x = [0.0, 1.0, 2.0]
y = 12.6667


Writing the intermediate tensors makes the graph visible in the code. Both expressions must produce the same result.

In [2]:
a = x + 2
b = a.square()
c = b + 3
y_by_steps = c.mean()
assert torch.allclose(y, y_by_steps)
print(f"a = {a.detach().tolist()}")
print(f"b = {b.detach().tolist()}")
print(f"c = {c.detach().tolist()}")

a = [2.0, 3.0, 4.0]
b = [4.0, 9.0, 16.0]
c = [7.0, 12.0, 19.0]


## Check autograd against the derivative

`x` is a leaf tensor with `requires_grad=True`, so after the backward pass its `.grad` contains $\partial y/\partial x_i$. The numerical comparison is a useful check that I have followed every step of the chain rule.

In [3]:
y.backward()
by_hand = 2 * (x.detach() + 2) / len(x)
assert torch.allclose(x.grad, by_hand)
for value, automatic, manual in zip(x.detach(), x.grad, by_hand):
    print(f"x={value.item():.0f}  autograd={automatic.item():.4f}  by hand={manual.item():.4f}")

x=0  autograd=1.3333  by hand=1.3333
x=1  autograd=2.0000  by hand=2.0000
x=2  autograd=2.6667  by hand=2.6667


## The same idea in one neuron

A neuron first computes $z = w^\mathsf{T}x+b$ and then applies an activation $o=\tanh(z)$. For fixed inputs, the chain rule gives

$$\frac{\partial o}{\partial w_i}=(1-o^2)x_i,\qquad
  \frac{\partial o}{\partial b}=1-o^2.$$

This example checks those gradients for one neuron; it does not train a network.

In [4]:
inputs = torch.tensor([2.0, 1.0], dtype=torch.float64)
weights = torch.tensor([0.3, -0.2], dtype=torch.float64, requires_grad=True)
bias = torch.tensor(0.1, dtype=torch.float64, requires_grad=True)
output = torch.tanh(weights @ inputs + bias)
output.backward()

slope = 1 - output.detach().square()
manual_weights = slope * inputs
manual_bias = slope
assert torch.allclose(weights.grad, manual_weights)
assert torch.allclose(bias.grad, manual_bias)
print(f"output = {output.item():.4f}")
print(f"weight gradients: autograd={weights.grad.tolist()}, by hand={manual_weights.tolist()}")
print(f"bias gradient: autograd={bias.grad.item():.4f}, by hand={manual_bias.item():.4f}")

output = 0.4621
weight gradients: autograd=[1.5728954659318548, 0.7864477329659274], by hand=[1.5728954659318548, 0.7864477329659274]
bias gradient: autograd=0.7864, by hand=0.7864


## Next step

The next exercise is to add a loss function and update the weights on a small classification dataset. That will turn this derivative check into a training loop. For the mechanics behind `requires_grad` and `backward()`, see the [PyTorch autograd tutorial](https://docs.pytorch.org/tutorials/beginner/basics/autograd_tutorial.html).